In [1]:
import pandas as pd
import clickhouse_connect
from datetime import datetime





import os
from dotenv import load_dotenv

load_dotenv()


True

In [2]:

# создаём клиент
# client = clickhouse_connect.get_client(
#     host= os.getenv("CLICKHOUSE_HOST") ,   # замени на свой хост/адрес
#     port= os.getenv("CLICKHOUSE_PORT"),
#     username=  os.getenv("CLICKHOUSE_USER"),
#     password= os.getenv("CLICKHOUSE_PASSWORD")
# )
client = clickhouse_connect.get_client(
    host="84.201.160.255",   # замени на свой хост/адрес
    port=8123,
    username="peter",
    password="1234"
)

In [3]:
# SQL-запрос
query = "SELECT * FROM mailkb.emails ORDER BY id DESC LIMIT 5"

# вернуть сразу DataFrame
df = client.query_df(query)


In [5]:
df

,id,message_id,subject,from_addr,to_addr,cc_addr,bcc_addr,sent_at_utc,sent_at_raw,folder,body_text,body_html
0,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,"RE: Запуск SAP: 6 Филиалов, график координаци...",[Maxim.Lyutov@ru.ey.com],"[Ilya.Sergeev@ru.ey.com, anton.chigvintsev@bea...",[],[],2018-08-19 11:48:48,"Sun, 19 Aug 2018 11:48:48 +0000",unknown,Это МТЗ?\r\n________________________________\r...,"<html><head>\r\n<meta http-equiv=""Content-Type..."
1,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,"RE: Запуск SAP: 6 Филиалов, график координаци...",[Maxim.Lyutov@ru.ey.com],"[anton.chigvintsev@bearingpoint.ru, ce-rustam....",[],[],2018-08-19 12:54:33,"Sun, 19 Aug 2018 12:54:33 +0000",unknown,"Илья, Антн,\r\nдобавьте статус гашения Всд в д...","<html><head>\r\n<meta http-equiv=""Content-Type..."
2,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,"RE: Запуск SAP: 6 Филиалов, график координаци...",[Maxim.Lyutov@ru.ey.com],"[anton.chigvintsev@bearingpoint.ru, ce-yuriy.k...",[],[],2018-08-19 11:38:13,"Sun, 19 Aug 2018 11:38:13 +0000",unknown,"Илья Сергеев, проверь пожалуйста наличие всд в...","<html><head>\r\n<meta http-equiv=""Content-Type..."
3,archive.pst::3027620,,,[],[],[],[],1970-01-01 00:00:00,,unknown,szeligowska_ea@segezha-group.com 16:27: \r\nПё...,"<html xmlns=""http://schemas.microsoft.com/2008..."
4,archive.pst::3024932,,,[],[],[],[],1970-01-01 00:00:00,,unknown,Пропущенный звонок конференц-связи от пользова...,"<html xmlns=""http://schemas.microsoft.com/2008..."


In [9]:
## импортируем либы для структурированного ответа

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from typing import List



class MailInfo(BaseModel):
    """Contact information for a person."""
    topic: str = Field(..., description="The name of the mail ")
    email: str = Field(..., description="The email address of the persons of the mail")

class MailChainInfo(BaseModel):
    emails: List[MailInfo] = Field(description='mail chain info')

agent = create_agent(
    model="gpt-5",
    response_format=MailChainInfo  # Auto-selects ProviderStrategy
)

# result = agent.invoke({
#     "messages": [{"role": "user", "content": "Extract contact info from: {}"}]
# })
#
# print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [7]:
df['body_text'][0]

'Это МТЗ?\r\n________________________________\r\n\r\nОт: Ilya M Sergeev <Ilya.Sergeev@ru.ey.com>\r\nОтправлено: воскресенье, 19 августа 2018 г. 14:47\r\nКому: Maxim P Lyutov <Maxim.Lyutov@ru.ey.com>,Chigvintsev Anton <anton.chigvintsev@bearingpoint.ru>,"Kozlov Yuriy(CE)" <ce-yuriy.kozlov@bearingpoint.ru>,Алексей Алмакаев <A.Almakaev@agrohold.ru>,Zhanna L Vakhrusheva <Zhanna.L.Vakhrusheva@ru.ey.com>,Алексей Филимончук <A.Filimonchuk@agrohold.ru>,Андрей Кравченко <A.Kravchenko@agrohold.ru>,Андрей Николаевич Петров <A.N.Petrov@agrohold.ru>,Yuri A Denisov <Yuri.Denisov@ru.ey.com>,Dmitry V Karpov <Dmitry.Karpov@ru.ey.com>,Alexey S Fedoseev <Alexey.Fedoseev@ru.ey.com>,Валерий Бражников <V.Brazhnikov@agrohold.ru>,Vadim Kotenko <V.Kotenko@agrohold.ru>,"Scherbakov Andrey(CE)" <ce-andrey.scherbakov@bearingpoint.ru>,Андрей Владимирович Самойленко <AV.Samoylenko@agrohold.ru>,Вадим Просолупов <V.Prosolupov@agrohold.ru>,Александр Игоревич Александров <a.i.aleksandrov@agrohold.ru>,Иван Лосев <I.Losev

In [10]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""Начни обработку строки и вытащи темы писем и email адреса:

{df['body_text'][0]}   """

        }
    ]
})

report = result["messages"][-1].content
print(report)



{"emails":[{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Ilya.Sergeev@ru.ey.com"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Maxim.Lyutov@ru.ey.com"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"anton.chigvintsev@bearingpoint.ru"},{"topic":"Re: Запуск SAP: 6 Филиалов, график координации перехода","email":"ce-yuriy.kozlov@bearingpoint.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"A.Almakaev@agrohold.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"A.Filimonchuk@agrohold.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"A.Kravchenko@agrohold.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Yuri.Denisov@ru.ey.com"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Alexey.Fedoseev@ru.ey.com"},{"topic":"Re: Запуск SAP: 6 Филиалов, график координ

In [12]:
Structured output


ключевые поинты
emailы все

SyntaxError: invalid syntax (734425773.py, line 1)